In [1]:
!pip install -q transformers datasets torch sentencepiece accelerate

In [2]:
from datasets import load_dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split='train[:10]')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "facebook/nllb-200-distilled-600M"
device = "cuda" if torch.cuda.is_available() else "cpu"


tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def translate_text(text, tgt_lang="mkd_Cyrl"):

    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)


    forced_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_id,
        max_length=512
    )

    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

print("Моделот е подготвен!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Моделот е подготвен!


In [4]:
def process_batch(example):

    try:
        example["instruction_mk"] = translate_text(example["instruction"])
        example["response_mk"] = translate_text(example["response"])
    except Exception as e:
        print(f"Грешка кај еден ред: {e}")
        example["instruction_mk"] = "ГРЕШКА"
        example["response_mk"] = "ГРЕШКА"
    return example

print("Започнувам со преведување на 10 примери...")
translated_dataset = dataset.map(process_batch)
print("Готово!")

Започнувам со преведување на 10 примери...


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Готово!


In [5]:
import pandas as pd
df = translated_dataset.to_pandas()
display(df[['instruction', 'instruction_mk', 'response', 'response_mk']].head(20))

,instruction,instruction_mk,response,response_mk
0,When did Virgin Australia start operating?,Кога започнал да работи Вирџин Австралија?,Virgin Australia commenced services on 31 Augu...,Виргин Австралија започна со услуги на 31 авгу...
1,Which is a species of fish? Tope or Rope,Кој вид на риба е?,Tope,Топеа
2,Why can camels survive for long without water?,Зошто камилите можат да преживеат долго без вода?,Camels use the fat in their humps to keep them...,Камила ги користи мазнините во своите голуби з...
3,"Alice's parents have three daughters: Amy, Jes...","Родителите на Алис имаат три ќерки: Ејми, Џеси...",The name of the third daughter is Alice,Третата ќерка се вика Алиса.
4,When was Tomoaki Komorida born?,Кога се родил Томоаки Коморида?,"Tomoaki Komorida was born on July 10,1981.",Tomoaki Komorida се роди на 10 јули 1981 година.
5,If I have more pieces at the time of stalemate...,Ако имам повеќе парчиња во моментот на заглави...,No. \nStalemate is a drawn position. It doesn'...,"Не, сталемата е позиција на течење, не е важно..."
6,"Given a reference text about Lollapalooza, whe...","Со оглед на референтен текст за Lollapalooza, ...",Lollapalooze is an annual musical festival hel...,Lollapalooze е годишен музички фестивал кој се...
7,Who gave the UN the land in NY to build their HQ,Кој му дал на ОН земјата во Њујорк за да ја из...,John D Rockerfeller,Џон Д. Рокерфелер
8,Why mobile is bad for human,Зошто мобилниот е лош за човекот?,We are always engaged one phone which is not g...,"Секогаш сме ангажирани со еден телефон, што не..."
9,Who was John Moses Browning?,Кој беше Џон Мојсеј Браунинг?,John Moses Browning is one of the most well-kn...,Џон Мојсеј Браунинг е еден од најпознатите диз...
